# SQL query from table names

In This notebook we are going to test if using just the name of the table, and a shord definition of its contect we can use a model like GTP3.5-Turbo to select which tables are necessary to create a SQL Order to answer the user petition.

In [25]:
from openai import OpenAI
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')

In [26]:
#Function to call the model.
def return_OAI(user_message):
    client = OpenAI(
    # This is the default and can be omitted
    api_key=OPENAI_API_KEY,
)
    context = []
    context.append({'role':'system', "content": user_message})

    response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=context,
            temperature=0,
        )

    return (response.choices[0].message.content)

In [27]:
#Definition of the tables.
import pandas as pd

# Table and definitions sample
data = {'table': ['rider', 'team', 'circuit', 'year', 'championships'],# ENTER A TABLE COLUMNS HERE,
        'definition': ['Rider information, name...',
                       'Group supporting a rider',
                       'Track where races occur', 
                       'Calendar year of the race',
                       'Total career championship titles'],
        'sample_data': [
        'name: Marc Márquez, nationality: Spanish', 
        'name: Repsol Honda, country: Japan',
        'name: Silverstone, location: UK',
        'year: 2023, 2024, 2025',
        'titles: 8, category: MotoGP'
    ]          
        } 
                                # ENTER A TABLE DEFINITATIONS HERE
df = pd.DataFrame(data)
print(df)

           table                        definition  \
0          rider        Rider information, name...   
1           team          Group supporting a rider   
2        circuit           Track where races occur   
3           year         Calendar year of the race   
4  championships  Total career championship titles   

                                sample_data  
0  name: Marc Márquez, nationality: Spanish  
1        name: Repsol Honda, country: Japan  
2           name: Silverstone, location: UK  
3                    year: 2023, 2024, 2025  
4               titles: 8, category: MotoGP  


In [28]:
text_tables = '\n'.join([
    f"Table: {row['table']}\n - Definition: {row['definition']}\n - Example: {row['sample_data']}\n" 
    for index, row in df.iterrows()
])

print(text_tables)

Table: rider
 - Definition: Rider information, name...
 - Example: name: Marc Márquez, nationality: Spanish

Table: team
 - Definition: Group supporting a rider
 - Example: name: Repsol Honda, country: Japan

Table: circuit
 - Definition: Track where races occur
 - Example: name: Silverstone, location: UK

Table: year
 - Definition: Calendar year of the race
 - Example: year: 2023, 2024, 2025

Table: championships
 - Definition: Total career championship titles
 - Example: titles: 8, category: MotoGP



In [29]:
print(text_tables)

Table: rider
 - Definition: Rider information, name...
 - Example: name: Marc Márquez, nationality: Spanish

Table: team
 - Definition: Group supporting a rider
 - Example: name: Repsol Honda, country: Japan

Table: circuit
 - Definition: Track where races occur
 - Example: name: Silverstone, location: UK

Table: year
 - Definition: Calendar year of the race
 - Example: year: 2023, 2024, 2025

Table: championships
 - Definition: Total career championship titles
 - Example: titles: 8, category: MotoGP



In [30]:
prompt_question_tables = """Given the following tables, their definitions, and sample data:

### Tables
{tables}

Tell me which tables are necessary to answer the user's question. 
Use the sample data to confirm you are choosing the right columns.

Return the table names in a JSON format.

### User Question:
{question}"""

# Prueba con Marc Márquez
pqt_marquez = prompt_question_tables.format(
    tables=text_tables, 
    question="How many championships does Marc Márquez have?"
)

print(return_OAI(pqt_marquez))

{
    "tables": ["rider", "championships"]
}


In [31]:
#Creating the prompt, with the user questions and the tables definitions.
pqt1 = prompt_question_tables.format(tables=text_tables, question="Which 3 riders have won the most races in the last 5 years?")

In [32]:
print(return_OAI(pqt1))

{
    "tables": {
        "rider": ["name"],
        "year": ["year"]
    }
}


In [33]:
#Creating the prompt, with the user questions and the tables definitions.
pqt2 = prompt_question_tables.format(tables=text_tables, question="Which 3 teams have won the most championships in the last 10 years?")

In [34]:
print(return_OAI(pqt2))

{
    "tables": ["team", "year", "championships"]
}


In [35]:
pqt3 = prompt_question_tables.format(tables=text_tables,
                                     question="Which rider has the most championships of all time?")#ENTER YOUR QUERY HERE

In [36]:
print(return_OAI(pqt3))

{
    "tables": ["rider", "championships"]
}


# Exercise
 - Complete the prompts similar to what we did in class. 
     - Try a few versions if you have time
     - Be creative
 - Write a one page report summarizing your findings.
     - Were there variations that didn't work well? i.e., where GPT either hallucinated or wrong
 - What did you learn?

# 📊 Technical Report: Semantic Table Selection for Text-to-SQL Pipelines

## 1. Summary of Findings
In this notebook, we evaluated the capability of **GPT-3.5-Turbo** to identify and select the necessary database tables to answer natural language questions. Instead of providing a full DDL (Data Definition Language) schema, we used a **semantic metadata approach**.

**Key Results:**
* The model successfully mapped user intents to table names using only short definitions.
* The inclusion of **Sample Data** (e.g., `name: Marc Márquez`) significantly improved the "grounding" of the model, reducing ambiguity between entities.
* Output consistency was maintained by forcing a **JSON format**, essential for downstream automation in AI pipelines.

---

## 2. Variations & Hallucination Analysis
During the experimentation, we observed how the model behaves under different prompt variations:

* **What didn't work well:**
    * **Initial Hallucinations:** Without sample data, the model occasionally over-selected tables (e.g., selecting `circuit` for questions about race counts, even if track details weren't needed).
    * **Semantic Confusion:** In cases with similar table names (e.g., `championships` vs `rider` stats), the model required a more granular definition to avoid choosing the wrong source.
* **What worked best:** * Providing a **concrete example** within the metadata (e.g., `name: Marc Márquez`) acted as a "semantic anchor", preventing the model from guessing the table content.

---

## 3. Key Learnings
As an **AI Engineer**, this exercise provided the following insights:

1. **Metadata Engineering:** The quality of the output is 100% dependent on the clarity of the table definitions. Well-curated metadata can replace a complex schema DDL for selection tasks.
2. **Context Optimization:** Pre-selecting tables is an effective strategy to manage the **Context Window** of LLMs, allowing the system to scale to databases with hundreds of tables without exceeding token limits.
3. **Few-Shot Power:** Small "shadowing" of data (showing one real record) is more effective than long descriptive instructions for grounding the LLM in a specific domain (like MotoGP).

---

## 4. Final Conclusion
The experiment proves that a **Metadata-First** approach is a robust and cost-effective method for the initial stage of a Text-to-SQL pipeline. By using JSON outputs and deterministic settings (`temperature=0`), we can build a reliable bridge between human questions and relational databases.